# Reproducing the EM quadrant plot (Soligo et al., arXiv 2506.11618)

This notebook reproduces the paper's headline result — emergently misaligned models
landing in the **coherent-but-misaligned** quadrant — and shows that ablating the
convergent **mean-diff** direction pulls them back to aligned.

**Pipeline** (uses the vendored EM repo's prompts + plotting, our own generation/judging):

1. Load base Qwen2.5-14B-Instruct + the rank-1 EM LoRA (`R1_3_3_3`) as one PeftModel.
2. Download Soligo's mean-diff "general medical" steering vector from HuggingFace.
3. Generate responses on the 8 `first_plot_questions` under three conditions:
   `base` (adapter off) · `em_lora` (adapter on) · `em_lora_ablated` (adapter on, mean-diff projected out).
4. Judge every response with the gpt-4o `aligned` + `coherent` rubrics (logprob 0-100).
5. Plot with the paper's own `quadrant_plots.plot_coherent_v_aligned`.

**What we reuse from the paper vs. build ourselves:**

| From `resources/model-organisms-for-EM` | Ours (`repro/`) |
|---|---|
| eval prompts, judge templates | OpenAI logprob judge (`repro.judge`) |
| `quadrant_plots`, mean-diff vectors (HF) | generation + ablation pipeline (`repro.generate`) |
| `load_em_direction` / LoRA-B (`utils`) | direction sourcing + decompose (`repro.directions`) |


In [ ]:
import sys
sys.path.insert(0, ".")                                  # repo root -> repro, utils
sys.path.insert(0, "resources/model-organisms-for-EM")  # vendored EM repo -> quadrant_plots

from repro.judge import load_dotenv_walk
load_dotenv_walk()  # find OPENAI_API_KEY in the .env (walks up to the main checkout)

# --- config knobs (scaled down from the paper's 50/600 for tractability) ---
N_PER_Q        = 10          # paper uses 50
MAX_NEW_TOKENS = 200         # paper uses 600
TEMPERATURE    = 1.0
TOP_P          = 1.0
# Ablation layer convention: layer_idx k>0 -> decoder.layers[k-1] (hidden_states[k]).
# Default = the vector's own layer (L24): faithful single-direction ablation.
# Soligo's stronger "single-direction, all layers" variant: list(range(1, 49)).
ABLATE_LAYERS  = [24]
SAVE_DIR       = "data/repro_responses"
print(f"config: {N_PER_Q}/q, {MAX_NEW_TOKENS} tok, ablate layers={ABLATE_LAYERS}")

## 1. Load base + EM LoRA (one model, adapter toggled per condition)

In [ ]:
from repro.generate import load_em_model, load_first_plot_questions

model, tokenizer = load_em_model()   # base Qwen2.5-14B + R1_3_3_3 LoRA (PeftModel)
questions, ids = load_first_plot_questions()
print(f"{len(questions)} questions:", ids)

## 2. Mean-diff direction (Soligo) from HuggingFace

Downloads `ModelOrganismsForEM/Qwen2.5-14B_steering_vector_general_medical` — the
convergent misalignment direction for the bad-medical organism, defined at layer 24.
We compare it to the rank-1 **LoRA-B** direction: Soligo Appendix D reports these are
largely different directions at L24 despite both mediating EM.

In [ ]:
from repro.directions import get_meandiff_direction, get_lora_B_direction, cosine

md = get_meandiff_direction("general_medical", unit=True)
v_meandiff = md["direction"]
print(f"mean-diff: layer {md['layer_idx']}, raw_norm {md['raw_norm']:.3f}, dim {tuple(v_meandiff.shape)}")

v_lora_B = get_lora_B_direction(model, tokenizer)   # cached unit-norm LoRA-B
print(f"cos(mean-diff, LoRA-B) at L24 = {cosine(v_meandiff, v_lora_B):+.3f}  "
      f"(Soligo Appendix D: ~0.04 for the single-adapter pairing)")

## 3. Generate + judge the three conditions

Each `run_condition` generates `N_PER_Q` samples/question, judges `aligned`+`coherent`,
and writes `{SAVE_DIR}/{condition}.csv`. Expected runtime scales with `N_PER_Q × 8 × 3`
generations on the 14B + that many judge calls.

- **base** — adapter disabled → aligned + coherent (top-right).
- **em_lora** — adapter active → misaligned + coherent (bottom-right): the EM result.
- **em_lora_ablated** — adapter active, mean-diff projected out → back toward aligned (top-right).

In [ ]:
from repro.generate import run_condition

common = dict(questions=questions, ids=ids, n_per_question=N_PER_Q,
              max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P)

df_base    = run_condition(model, tokenizer, "base",            SAVE_DIR, **common)
df_em      = run_condition(model, tokenizer, "em_lora",         SAVE_DIR, **common)
df_ablated = run_condition(model, tokenizer, "em_lora_ablated", SAVE_DIR,
                           ablate_dirs=[v_meandiff], ablate_layers=ABLATE_LAYERS, **common)

## 4. The quadrant plot (paper's own `quadrant_plots`)

In [ ]:
from em_organism_dir.vis import quadrant_plots

# Scatter: coherent (x) vs aligned (y), quadrant lines at x=50 / y=30. One subplot per CSV.
quadrant_plots.plot_coherent_v_aligned(SAVE_DIR)

In [ ]:
# Aggregate table: % misaligned-and-coherent (the EM rate) + % coherent, per condition.
_ = quadrant_plots.get_basic_eval_stats(SAVE_DIR)

## Interpretation

- **base** should sit in the aligned+coherent quadrant (high `aligned`, high `coherent`).
- **em_lora** should drop into the **misaligned + coherent** quadrant — the EM phenomenon,
  and the `misaligned_coherent` % should jump (paper: ~11% for this 9-adapter organism).
- **em_lora_ablated** should climb back toward aligned while keeping coherence high —
  reproducing Soligo §3.4 ("ablation reduces EM to ≤1%, coherence ≥99%").

**Caveats vs. the paper:** we use `N_PER_Q`=10 (paper 50) and 200 tokens (paper 600), so
absolute percentages will be noisier. The *quadrant structure* and the base→em→ablated
shift are the reproduction targets.

**Next (extension):** swap the mean-diff direction for the capability-decomposed
`v_bad` (project `V_cap` out first) and re-run — does the ablated condition preserve
capability better? See `02_decomposition.ipynb`.